# CFPB Complaint Intelligence Analysis

## Business question

Where are complaint risks concentrated (products, issues, companies, and time), and what actions should a CX/compliance team prioritize next?

## 1. Setup

This notebook is designed to be reproducible from local project data. It uses project modules for schema normalization, feature engineering, and risk scoring.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cx_complaints.data.schema import normalize_complaints
from cx_complaints.features.text_features import add_text_features
from cx_complaints.modelling.risk import score_complaint_risk
from cx_complaints.preprocessing.text import add_clean_text_column

plt.style.use("ggplot")
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 25)

## 2. Load and prepare data

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected sample file at data/raw/cfpb_sample.csv. Run the fetch command first."
    )

raw = pd.read_csv(raw_path)
complaints = normalize_complaints(raw)
complaints = add_clean_text_column(complaints)
complaints = add_text_features(complaints)
complaints = score_complaint_risk(complaints)

complaints.shape

In [ ]:
complaints[[
    "complaint_id",
    "date_received",
    "product",
    "issue",
    "company",
    "risk_score",
    "risk_level",
]].head()

## 3. Topline KPIs

In [ ]:
total = len(complaints)
high_risk = int((complaints["risk_level"] == "high").sum())
medium_or_high = int(complaints["risk_level"].isin(["medium", "high"]).sum())
timely_yes = int((complaints["timely"].fillna("").str.lower() == "yes").sum())

kpis = pd.DataFrame(
    {
        "metric": [
            "Total complaints",
            "High-risk complaints",
            "Medium/high-risk complaints",
            "Timely responses",
        ],
        "value": [
            total,
            high_risk,
            medium_or_high,
            timely_yes,
        ],
        "share_of_total_pct": [
            100.0,
            round(high_risk / max(total, 1) * 100, 1),
            round(medium_or_high / max(total, 1) * 100, 1),
            round(timely_yes / max(total, 1) * 100, 1),
        ],
    }
)
kpis

## 4. Concentration analysis (product, issue, company)

In [ ]:
def concentration_table(frame: pd.DataFrame, column: str, top_n: int = 10) -> pd.DataFrame:
    grouped = (
        frame.groupby(column, dropna=False)
        .agg(
            complaints=(column, "size"),
            avg_risk_score=("risk_score", "mean"),
            high_risk_share_pct=("risk_level", lambda s: (s == "high").mean() * 100),
        )
        .reset_index()
        .sort_values(["complaints", "avg_risk_score"], ascending=[False, False])
        .head(top_n)
    )
    grouped["avg_risk_score"] = grouped["avg_risk_score"].round(2)
    grouped["high_risk_share_pct"] = grouped["high_risk_share_pct"].round(1)
    return grouped

top_products = concentration_table(complaints, "product")
top_issues = concentration_table(complaints, "issue")
top_companies = concentration_table(complaints, "company")

top_products

In [ ]:
top_issues

In [ ]:
top_companies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

product_share = complaints["product"].fillna("Unknown").value_counts().head(10)
issue_share = complaints["issue"].fillna("Unknown").value_counts().head(10)

axes[0].barh(product_share.index[::-1], product_share.values[::-1], color="#2F6690")
axes[0].set_title("Top 10 Products by Complaint Volume")
axes[0].set_xlabel("Complaint count")

axes[1].barh(issue_share.index[::-1], issue_share.values[::-1], color="#3A7D44")
axes[1].set_title("Top 10 Issues by Complaint Volume")
axes[1].set_xlabel("Complaint count")

fig.tight_layout()

## 5. Trend analysis over time

In [ ]:
complaints["date_received"] = pd.to_datetime(complaints["date_received"], errors="coerce")
monthly = (
    complaints.dropna(subset=["date_received"])
    .assign(month=lambda d: d["date_received"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(
        complaints=("complaint_id", "size"),
        avg_risk_score=("risk_score", "mean"),
        high_risk_share_pct=("risk_level", lambda s: (s == "high").mean() * 100),
    )
    .reset_index()
)
monthly["avg_risk_score"] = monthly["avg_risk_score"].round(2)
monthly["high_risk_share_pct"] = monthly["high_risk_share_pct"].round(1)
monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(monthly["month"], monthly["complaints"], marker="o", color="#1D3557", label="Complaints")
ax2.plot(
    monthly["month"],
    monthly["high_risk_share_pct"],
    marker="s",
    color="#B22222",
    label="High-risk share (%)",
)

ax1.set_title("Monthly Complaint Volume and High-Risk Share")
ax1.set_xlabel("Month")
ax1.set_ylabel("Complaint count")
ax2.set_ylabel("High-risk share (%)")
ax1.tick_params(axis="x", rotation=45)

line1, label1 = ax1.get_legend_handles_labels()
line2, label2 = ax2.get_legend_handles_labels()
ax1.legend(line1 + line2, label1 + label2, loc="upper left")
fig.tight_layout()

## 6. Response quality and risk profile

In [ ]:
timely_table = (
    complaints.assign(timely_clean=complaints["timely"].fillna("Unknown").astype(str).str.title())
    .groupby("timely_clean")
    .agg(
        complaints=("complaint_id", "size"),
        avg_risk_score=("risk_score", "mean"),
        high_risk_share_pct=("risk_level", lambda s: (s == "high").mean() * 100),
    )
    .reset_index()
    .sort_values("complaints", ascending=False)
)
timely_table["avg_risk_score"] = timely_table["avg_risk_score"].round(2)
timely_table["high_risk_share_pct"] = timely_table["high_risk_share_pct"].round(1)
timely_table

In [ ]:
risk_distribution = complaints["risk_level"].fillna("unknown").str.lower().value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(
    risk_distribution.index,
    risk_distribution.values,
    color=["#B22222", "#E09F3E", "#4C9F70", "#7A7A7A"],
)
axes[0].set_title("Risk Level Distribution")
axes[0].set_xlabel("Risk level")
axes[0].set_ylabel("Complaint count")

axes[1].hist(complaints["risk_score"].dropna(), bins=20, color="#2F6690", edgecolor="white")
axes[1].set_title("Risk Score Distribution")
axes[1].set_xlabel("Risk score")
axes[1].set_ylabel("Complaint count")

fig.tight_layout()

## 7. Evidence-based recommendations

Use computed outputs above to fill this section after running the notebook:

1. **What changed**: reference the monthly chart and trend table.
2. **Where concentration is highest**: reference top products/issues/companies tables.
3. **Operational action**: prioritize high-risk complaints and untimely-response segments.
4. **Evidence**: cite specific table rows and percentages from this notebook.

In [ ]:
# Optional export for stakeholders
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

top_products.to_csv(output_dir / "notebook_top_products.csv", index=False)
top_issues.to_csv(output_dir / "notebook_top_issues.csv", index=False)
top_companies.to_csv(output_dir / "notebook_top_companies.csv", index=False)
monthly.to_csv(output_dir / "notebook_monthly_trends.csv", index=False)
timely_table.to_csv(output_dir / "notebook_timely_summary.csv", index=False)

output_dir